In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

import math
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import folium
from folium import plugins
from PIL import Image

from tqdm.notebook import tqdm
import os, sys
from io import StringIO, BytesIO
import glob
import urllib
import lxml.etree

import gpxpy

dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: sys.path.append(dir1)
from util.tcx_reader import TCXReader

In [ ]:
gpx_files = glob.glob("/home/jeppe/OneDrive/Data/running/*.gpx")
tcx_files = glob.glob("/home/jeppe/OneDrive/Data/running/*.tcx")

# TCX files

In [ ]:
tcx_dfs = []
tcx_reader = TCXReader()
for idx, tcx_file in enumerate(tcx_files):
    try:
        tcx_reader.read_file(tcx_file)
        laps_df, points_df = tcx_reader.get_dataframes()
        tcx_dfs.append(points_df)
    except Exception as e:
        print(e)

In [ ]:
points_df_all = pd.concat(tcx_dfs)
map = folium.Map(location=points_df_all.iloc[0][["latitude", "longitude"]], tiles="OpenStreetMap", zoom_start=13)

heat_data = [[point.xy[1][0], point.xy[0][0]] for point in points_df_all.geometry]

heat_data
plugins.HeatMap(heat_data).add_to(map)

map

In [ ]:
tcx_file = TCXReader()
tcx_file.read_file(tcx_files[-1])
laps_df, points_df = tcx_file.get_dataframes()

In [ ]:
map = folium.Map(location=points_df.iloc[0][["latitude", "longitude"]], tiles="OpenStreetMap", zoom_start=13)

heat_data = [[point.xy[1][0], point.xy[0][0]] for point in points_df.geometry]

heat_data
plugins.HeatMap(heat_data).add_to(map)

map

In [ ]:
map = folium.Map(location=points_df.iloc[0][["latitude", "longitude"]], tiles="OpenStreetMap", zoom_start=13)

points_df.apply(lambda row:folium.CircleMarker(location=[row["latitude"], row["longitude"]], 
                                              radius=1, popup=row['speed'])
                                             .add_to(map), axis=1)

map

In [ ]:
map = folium.Map(location=points_df.iloc[0][["latitude", "longitude"]], 
                #  tiles="OpenStreetMap", 
                 zoom_start=13)

loc = [(x["longitude"], x["latitude"]) for _, x in points_df.iterrows()]

folium.PolyLine(loc,
                color='red',
                weight=9,
                opacity=1).add_to(map)
map

# GPX files

In [ ]:
df = pd.DataFrame(
    columns=["latitude", "longitude", "time", "run_number"]
)

In [ ]:
for idx, gpx_file in tqdm(enumerate(gpx_files[:1])):
    gpx_file = open(gpx_file, 'r')
    gpx = gpxpy.parse(gpx_file)
    df_sub = pd.DataFrame(
        data=[[x.latitude, x.longitude, x.time, idx] for x in gpx.tracks[0].segments[0].points],
        columns=["latitude", "longitude", "time", "run_number"]
    )
    df = pd.concat([df, df_sub], ignore_index=True)

In [ ]:
df

In [ ]:
m = folium.Map([df["latitude"].min(), df["longitude"].min()], zoom_start=12)
#folium.GeoJson(nReserve).add_to(m)
m

In [ ]:
sns.scatterplot(data=df.sort_values("time"),
             x="longitude",
             y="latitude",
             hue="time",
             palette="viridis"
            )
plt.legend('')

In [ ]:
def deg2num(lat_deg, lon_deg, zoom):
  lat_rad = math.radians(lat_deg)
  n = 2.0 ** zoom
  xtile = int((lon_deg + 180.0) / 360.0 * n)
  ytile = int((1.0 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
  return (xtile, ytile)
  
def num2deg(xtile, ytile, zoom):
  n = 2.0 ** zoom
  lon_deg = xtile / n * 360.0 - 180.0
  lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * ytile / n)))
  lat_deg = math.degrees(lat_rad)
  return (lat_deg, lon_deg)
  
  
    
def getImageCluster(lat_deg, lon_deg, delta_lat,  delta_long, zoom):
    smurl = r"http://a.tile.openstreetmap.org/{0}/{1}/{2}.png"
    xmin, ymax = deg2num(lat_deg, lon_deg, zoom)
    xmax, ymin = deg2num(lat_deg + delta_lat, lon_deg + delta_long, zoom)
    
    Cluster = Image.new('RGB',((xmax-xmin+1)*256-1,(ymax-ymin+1)*256-1) ) 
    for xtile in range(xmin, xmax+1):
        for ytile in range(ymin,  ymax+1):
            try:
                imgurl=smurl.format(zoom, xtile, ytile)
                print("Opening: " + imgurl)
                imgreq = urllib.request.Request(
                    url=imgurl,
                    data=None,
                    headers={
                        "User-Agent": "Custom user agent for hobby project",
                        "From": "jeppe.t.kristensen@gmail.com"
                    }
                )
                imgstr = urllib.request.urlopen(imgreq).read()
                tile = Image.open(BytesIO(imgstr))
                Cluster.paste(tile, box=((xtile-xmin)*256, (ytile-ymin)*255))
            except Exception as e: 
                print("Couldn't download image")
                print(e)
                tile = None

    return Cluster
    
   
  
if __name__ == '__main__':
    
    a = getImageCluster(df["latitude"].min(), df["longitude"].min(), 0.02,  0.05, 13)
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    plt.imshow(np.asarray(a))
    plt.show()

In [ ]:
df[(df["longitude"]>15) & (df["latitude"]<45)]

In [ ]:
fig = plt.figure(figsize=(8, 8))
m = Basemap(projection='lcc', resolution='i', 
            lat_0=gpx.tracks[0].segments[0].points[0].latitude,
            lon_0=gpx.tracks[0].segments[0].points[0].longitude,
            width=1.05E4, height=1.2E4)
m.shadedrelief(scale=10)
m.drawcoastlines()